# Exploratory Data Analysis — Amazon All_Beauty Reviews

This notebook explores the dataset downloaded by `data/download_dataset.py`
(`data/reviews.csv`, 5 000 rows, McAuley-Lab/Amazon-Reviews-2023,
category `All_Beauty`).

**Goals**
1. Class distribution of star ratings.
2. Average review text length per rating.
3. Top-20 most frequent words (excluding stopwords).
4. Sample of 5 reviews per rating class.

Everything stays within pandas + matplotlib so there are no heavy NLP
dependencies at EDA time.

In [ ]:
# ---- Imports & config --------------------------------------------------
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Resolve the CSV path relative to the repo root so the notebook works
# whether it is launched from notebooks/ or from the project root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
CSV_PATH = PROJECT_ROOT / "data" / "reviews.csv"
print(f"Loading {CSV_PATH}")

In [ ]:
# ---- Load the dataset --------------------------------------------------
df = pd.read_csv(CSV_PATH)
print(f"Rows: {len(df):,}    Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# ---- Normalise the columns we need ------------------------------------
# The dataset uses `rating` (float 1-5) and `text` for the review body,
# but naming can drift between HuggingFace releases, so we defend against
# that here and fall back to sensible aliases.
rating_col = next((c for c in ["rating", "overall", "stars"] if c in df.columns), None)
text_col = next((c for c in ["text", "reviewText", "review_body"] if c in df.columns), None)
assert rating_col is not None, "No rating column found"
assert text_col is not None, "No review-text column found"

df[rating_col] = pd.to_numeric(df[rating_col], errors="coerce")
df = df.dropna(subset=[rating_col, text_col]).copy()
df["rating_int"] = df[rating_col].round().astype(int)
df["text_len"] = df[text_col].astype(str).str.len()
print(df[[rating_col, text_col, "rating_int", "text_len"]].head())

## (a) Class distribution of ratings

In [ ]:
counts = df["rating_int"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(counts.index.astype(str), counts.values, color="steelblue")
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of reviews")
ax.set_title("Class distribution of ratings (All_Beauty, 5k rows)")
for x, y in zip(counts.index.astype(str), counts.values):
    ax.text(x, y, f"{y:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()
counts

## (b) Average review text length per rating

In [ ]:
avg_len = df.groupby("rating_int")["text_len"].mean().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(avg_len.index.astype(str), avg_len.values, color="coral")
ax.set_xlabel("Star rating")
ax.set_ylabel("Average review length (characters)")
ax.set_title("Average review length per rating")
for x, y in zip(avg_len.index.astype(str), avg_len.values):
    ax.text(x, y, f"{y:,.0f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()
avg_len.round(1)

## (c) Top-20 most frequent words (excluding stopwords)

In [ ]:
# Small hand-rolled English stopword list — we intentionally avoid NLTK
# so the notebook has zero extra downloads at defence time.
STOPWORDS = set(
    """
    a about above after again against all am an and any are aren as at
    be because been before being below between both but by can cant could
    did do does doing dont down during each few for from further had has
    have having he her here hers herself him himself his how i if in into
    is it its itself just me more most my myself no nor not now of off
    on once only or other our ours ourselves out over own same she should
    so some such than that the their theirs them themselves then there
    these they this those through to too under until up very was we were
    what when where which while who whom why will with would you your
    yours yourself yourselves im ive dont didnt doesnt isnt wasnt werent
    thats theres get got one two also really very much still even way
    product products use used using item items make made little bit lot
    """.split()
)

TOKEN_RE = re.compile(r"[A-Za-z]{2,}")

def tokenise(text: str) -> list[str]:
    """Lowercase and split on runs of letters, then drop stopwords."""
    return [t for t in TOKEN_RE.findall(str(text).lower()) if t not in STOPWORDS]

counter: Counter[str] = Counter()
for text in df[text_col].astype(str):
    counter.update(tokenise(text))

top20 = counter.most_common(20)
top20_df = pd.DataFrame(top20, columns=["word", "count"])

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top20_df["word"][::-1], top20_df["count"][::-1], color="seagreen")
ax.set_xlabel("Frequency")
ax.set_title("Top-20 most frequent non-stopword tokens")
plt.tight_layout()
plt.show()
top20_df

## (d) Sample of 5 reviews per rating class

In [ ]:
SAMPLES_PER_CLASS = 5
for rating, group in df.groupby("rating_int"):
    sample = group.sample(min(SAMPLES_PER_CLASS, len(group)), random_state=42)
    print(f"\n==== Rating = {rating}  ({len(group)} reviews) ====")
    for i, row in enumerate(sample.itertuples(), 1):
        text = str(getattr(row, text_col)).replace("\n", " ")
        print(f"  [{i}] {text[:300]}{'…' if len(text) > 300 else ''}")